# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8

## A logistic regression model, built from parts

In the LELA60331 seminars you have built **logistic regression** step by step, as a long series of notebook cells, and in Week 7 you trained it on simulated data. In this workbook you will build the same classifier again, using what you learned about classes in Week 7, in a way that makes it easy to reuse and to extend. It has three parts:

| Part | What it does |
|---|---|
| a **layer** (the class `Linear`) | turns each example's features into a score (the **forward** pass), works out the gradients for its weights (the **backward** pass), and updates its weights |
| a **loss** (two functions, `bce_loss` and `bce_grad`) | measures how wrong the scores are, and says which way each score should move |
| a **training loop** | puts them together |

By the end of this workbook you will train the model with a loop like this one, in which each line does one job:

```python
for epoch in range(n_epochs):
    scores = model.forward(X)                 # the forward pass
    loss = bce_loss(scores, y)                # how wrong is it?
    model.backward(bce_grad(scores, y))       # the backward pass: the gradients
    model.update(lr)                          # change the weights
```

| Part | Topic |
|---|---|
| 1 | From a script to a class |
| 2 | The data as a matrix |
| 3 | A linear layer |
| 4 | The loss |
| 5 | The training loop |
| 6 | Using and evaluating the model |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · From a script to a class

In the LELA60331 seminars, each step of the classifier is a separate cell, and the cells share their results through **global names**. Even for the simulated data in LELA60331 Week 7 there are a lot of them:

| Name | What it holds |
|---|---|
| `x`, `y`, `X` | the features and the labels |
| `weights`, `bias` | the model's parameters |
| `lr`, `n_epochs` | the training settings |
| `q`, `errors`, `loss_history` | the probabilities, the errors and the losses |
| `score`, `sigmoid`, `average_loss`, `mat_vec`, ... | the functions that work on them |

This works, but it is fragile:

* Run the cells in the wrong order, or re-run one by mistake, and the names hold the wrong values. (In LELA60331 Week 7, `q` holds the probabilities from whichever cell you ran last: the untrained model or the trained one.)
* You can't easily have **two** models at once, because they would both want the name `weights`.
* To use the classifier on new data, you have to remember which cells to copy, and in which order.

A class solves these problems. Its **instance variables** keep the model's data together (its weights, its bias and their gradients), and its **methods** are the operations on them (Week 7). Each model is one object, with its own weights.

### 🏋️ Exercise 1 · Practice — Data or method?

The layer needs each of the things below. For each one, say whether it should be stored as an `"instance variable"` or written as a `"method"`.

In [ ]:
design = {
    "the weights": ???,
    "calculate the score of each example": ???,
    "the gradients for the weights": ???,
    "change the weights by gradient descent": ???,
}

In [ ]:
# ✅ Check
check("data", lambda: design["the weights"] == design["the gradients for the weights"] == "instance variable")
check("operations", lambda: design["calculate the score of each example"] == design["change the weights by gradient descent"] == "method")

---
# Part 2 · The data as a matrix

Here is the simulated data from LELA60331 Weeks 5 and 7: 50 texts, each with two features (the log counts of positive and of negative words) and a label (1 = positive, 0 = negative). As in the seminar, `x[0]` holds feature 1 for every text and `x[1]` holds feature 2. Run the cell.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
print(len(y), "texts,", sum(y), "of them positive")

In LELA60331 Week 7, Part 4 you saw that it is more convenient to store the data as a **matrix** `X` with **one row per text** and one column per feature, so that one matrix-vector multiplication gives the score of every text. You also saw that the gradients need the **transpose** of `X`: the same numbers with the rows turned into columns, one row per feature.

### 🏋️ Exercise 2 · Practice — Rows and the transpose

1. `X`: a list with one row `[x1, x2]` for each text (as in LELA60331 Week 7, Activities 6 and 7)
2. Complete `transpose(matrix)`, which returns a new matrix whose row `j` is column `j` of `matrix`. *Hint:* column `j` is `[row[j] for row in matrix]` (Week 4, Part 3), and there is one column for each element of the first row.

In [ ]:
X = ???                  # 1.

def transpose(matrix):
    return ???   # 2.

print(len(X), len(X[0]), X[0])
print(transpose([[1, 2, 3], [4, 5, 6]]))       # you should get [[1, 4], [2, 5], [3, 6]]

In [ ]:
# ✅ Check
check("1. X has 50 rows of 2", lambda: len(X) == 50 and all(len(row) == 2 for row in X) and X[0] == [x[0][0], x[1][0]])
check("2. transpose of a small matrix", lambda: transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]])
check("2. the transpose of X is x", lambda: transpose(X) == x)

Two more helper functions from LELA60331 do the arithmetic: `dot(a, b)` (Week 4) and `mat_vec(matrix, vector)` (Week 7, Activity 10). Run the cell; it also defines the `sigmoid` function from LELA60331 Week 5.

In [ ]:
def dot(a, b):
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

---
# Part 3 · A linear layer

A `Linear` layer does the matrix-vector multiplication from LELA60331 Week 7. It has `n_out` **outputs**, and for each example it gives one score per output. Logistic regression needs just one output: the score that the text is positive. (Other models need more, and that is why the layer allows several.)

**Its instance variables:**

* `self.weight`: a matrix with **one row per output** and one column per input feature, so `mat_vec(self.weight, x)` gives the scores of example `x` (without the biases)
* `self.bias`: a list with one bias per output
* both start at random numbers between 0 and 1, as in LELA60331 Week 7

**Its methods:**

* **`forward(X)`** takes a list of examples (one row each) and returns a matrix with one row per example and one score per output. It also keeps `X` in `self.inputs`, because the backward pass needs it.
* **`backward(grad)`** works out the gradients. `grad` has the same shape as the scores: `grad[k][c]` says how much the loss changes when the score of output `c` for example `k` goes up. Starting from gradients of 0, it adds each example's contribution, just as in LELA60331 Week 7:
  * the gradient for `weight[c][j]` gets `grad[k][c] * inputs[k][j]` from example `k` (the error times the feature)
  * the gradient for `bias[c]` gets `grad[k][c]` from example `k`

  (Its last line returns the gradients for the layer's **inputs**, which are needed when one layer feeds into another. It is done for you.)
* **`update(lr)`** changes every weight and bias: **new value = old value − learning rate × gradient**

### 🏋️ Exercise 3 · Practice — Complete `Linear`

Fill in the blanks 1–6. The tests underneath set the weights by hand, so that you can check the numbers.

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X                                    # kept for the backward pass
        scores = []
        for x in X:
            z = ???                  # 1. one score per output
            for c in range(len(z)):
                z[c] += ???                     # 2. plus that output's bias
            scores.append(z)
        return scores

    def backward(self, grad):
        n_out = len(self.weight)
        n_in = len(self.weight[0])
        self.weight_grad = [[0] * n_in for c in range(n_out)]     # start from 0s
        self.bias_grad = [0] * n_out
        for k in range(len(self.inputs)):                  # each example
            for c in range(n_out):                         # each output
                for j in range(n_in):                      # each input feature
                    self.weight_grad[c][j] += ???     # 3.
                self.bias_grad[c] += ???                                  # 4.
        # the gradients for the inputs, for a layer that comes before this one (done for you)
        return [mat_vec(transpose(self.weight), g) for g in grad]

    def update(self, lr):
        for c in range(len(self.weight)):
            for j in range(len(self.weight[c])):
                self.weight[c][j] -= ???                 # 5.
            self.bias[c] -= ???                               # 6.


layer = Linear(2, 3)
layer.weight = [[1, 0], [0, 1], [1, 1]]         # set the weights and biases by hand, to test it
layer.bias = [0, 10, 100]
print(layer.forward([[2, 3], [1, 1]]))           # you should get [[2, 13, 105], [1, 11, 102]]
layer.backward([[1, 0, 0], [0, 0, 2]])
print(layer.weight_grad, layer.bias_grad)        # you should get [[2, 3], [0, 0], [2, 2]] [1, 0, 2]
layer.update(0.5)
print(layer.weight, layer.bias)                  # you should get [[0.0, -1.5], [0.0, 1.0], [0.0, 0.0]] [-0.5, 10.0, 99.0]

In [ ]:
# ✅ Check
def _layer():
    l = Linear(2, 3)
    l.weight = [[1, 0], [0, 1], [1, 1]]
    l.bias = [0, 10, 100]
    return l
check("the shapes", lambda: len(Linear(4, 2).weight) == 2 and len(Linear(4, 2).weight[0]) == 4 and len(Linear(4, 2).bias) == 2)
check("1–2. forward", lambda: _layer().forward([[2, 3], [1, 1]]) == [[2, 13, 105], [1, 11, 102]])
def _back():
    l = _layer()
    l.forward([[2, 3], [1, 1]])
    l.backward([[1, 0, 0], [0, 0, 2]])
    return l.weight_grad == [[2, 3], [0, 0], [2, 2]] and l.bias_grad == [1, 0, 2]
check("3–4. backward", _back)
def _twice():
    l = _layer()
    l.forward([[2, 3], [1, 1]])
    l.backward([[1, 0, 0], [0, 0, 2]])
    l.backward([[1, 0, 0], [0, 0, 2]])
    return l.weight_grad == [[2, 3], [0, 0], [2, 2]]
check("3–4. backward starts from 0s each time", _twice)
def _update():
    l = _layer()
    l.forward([[2, 3], [1, 1]])
    l.backward([[1, 0, 0], [0, 0, 2]])
    l.update(0.5)
    return l.weight == [[0.0, -1.5], [0.0, 1.0], [0.0, 0.0]] and l.bias == [-0.5, 10.0, 99.0]
check("5–6. update", _update)

---
# Part 4 · The loss

The loss measures how wrong the model's scores are. For two classes we use the cross-entropy loss from LELA60331 Week 5 (sometimes called **binary cross-entropy**, or BCE). It turns each score into a probability with `sigmoid`, then takes the average of minus the log of the probability given to the correct label.

The backward pass needs one more thing from the loss: for each example, how much the loss changes when that example's score goes up. For this loss, that is exactly the **error** from LELA60331 Week 5, divided by the number of examples (because the loss is an average):

`(q - y[k]) / n`, where `q` is the sigmoid of example `k`'s score

`bce_grad` returns these as a matrix with one row per example and one number in each row: the same shape as the scores, so that it can be passed straight to the layer's `backward`.

### 🏋️ Exercise 4 · Practice — `bce_loss` and `bce_grad`

Complete the two functions. A layer with one output gives one row per example with one score in it, so the score of example `k` is `scores[k][0]`.

In [ ]:
def bce_loss(scores, y):
    eps = 0.00001          # a tiny number, so that we never take the log of 0
    losses = []
    for k in range(len(y)):
        q = ???                       # 1. the probability
        if y[k] == 1:
            losses.append(-math.log2(q + eps))
        else:
            losses.append(-math.log2(1 - q + eps))
    return ???                    # 2. the average

def bce_grad(scores, y):
    n = len(y)
    return ???     # 3.

print(round(bce_loss([[0.0], [2.0]], [1, 0]), 3))                          # you should get 2.034
print([[round(v, 3) for v in row] for row in bce_grad([[0.0], [2.0]], [1, 0])])   # you should get [[-0.25], [0.44]]

In [ ]:
# ✅ Check
check("1–2. bce_loss", lambda: round(bce_loss([[0.0], [2.0]], [1, 0]), 3) == 2.034)
check("3. bce_grad", lambda: [[round(v, 3) for v in row] for row in bce_grad([[0.0], [2.0]], [1, 0])] == [[-0.25], [0.44]])

---
# Part 5 · The training loop

Logistic regression is just a `Linear` layer with one output per example, trained with this loss. Each pass through the data (each **epoch**):

1. calculates the scores (the **forward pass**)
2. calculates the loss, and records it
3. calculates the gradients (the **backward pass**): `model.backward(bce_grad(scores, y))`
4. updates the weights and the bias

### 🏋️ Exercise 5 · Practice — The training loop

1. `model`: a `Linear` layer with 2 inputs and 1 output
2. Complete the training loop.

The settings are those of LELA60331 Week 7: a learning rate of 0.5 and 1000 epochs, with `random.seed(10)` before the model is made.

In [ ]:
random.seed(10)
model = ???                         # 1.
lr = 0.5
n_epochs = 1000
loss_history = []

for epoch in range(n_epochs):                  # 2.
    scores = ???
    loss = ???
    loss_history.append(loss)
    ???
    ???

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print(model.weight, model.bias)                # about [[1.17, -1.32]] and [0.51]
print("final loss:", round(loss_history[-1], 3))    # you should get 0.656

In [ ]:
# ✅ Check
check("1. one output, two inputs", lambda: len(model.weight) == 1 and len(model.weight[0]) == 2)
check("2. the loss went down", lambda: loss_history[-1] < loss_history[0])
check("2. the same weights as LELA60331 Week 7", lambda: [round(w, 2) for w in model.weight[0]] == [1.17, -1.32] and round(model.bias[0], 2) == 0.51)
check("2. final loss about 0.656", lambda: abs(loss_history[-1] - 0.656) < 0.001)

These are the weights from LELA60331 Week 7. Compare the two pieces of code: which lines in the seminar's training loop does each line of this loop replace?

---
# Part 6 · Using and evaluating the model

The model gives **scores**. To use it, we write three small functions:

* `predict_proba(model, X)`: the probability that each example is positive: the sigmoid of its score
* `predict(model, X)`: the predicted class of each example: 1 if its probability is above 0.5, and 0 otherwise
* `evaluate(model, X, y)`: a **dict** (Week 2) with the accuracy, precision and recall (LELA60331 Week 7, Activities 2 and 4). The counting is done for you.

They work for any model that has a `forward` method and gives one score per example, so they are kept outside the class.

### 🏋️ Exercise 6 · Practice — Utilities

1. Complete the three functions.
2. `results`: the trained model's results on `X` and `y`
3. `new_probs`: the probability that each of these two new texts is positive (LELA60331 Week 7, Activity 3). Remember that the features are **log** counts, and that the model takes a **list** of examples.
   * a) 2 positive words and 3 negative words
   * b) 10 positive words and 1 negative word

In [ ]:
def predict_proba(model, X):
    return ???          # 1.

def predict(model, X):
    return ???

def evaluate(model, X, y):
    predicted = ???
    correct = 0
    tp = 0
    fp = 0
    fn = 0
    for k in range(len(y)):
        if predicted[k] == y[k]:
            correct += 1
        if predicted[k] == 1 and y[k] == 1:
            tp += 1
        if predicted[k] == 1 and y[k] == 0:
            fp += 1
        if predicted[k] == 0 and y[k] == 1:
            fn += 1
    return {"accuracy": ???, "precision": ???, "recall": ???}

results = ???                                    # 2.
new_probs = ???    # 3.
print(results)
print(new_probs)                 # about 0.466 and 0.961

In [ ]:
# ✅ Check
check("1. predict_proba", lambda: abs(predict_proba(model, [[0, 0]])[0] - sigmoid(model.bias[0])) < 1e-12)
check("1. predict", lambda: predict(model, X[:5]) == [int(p > 0.5) for p in predict_proba(model, X[:5])])
check("2. accuracy 0.78, precision 0.75, recall 0.783",
      lambda: results["accuracy"] == 0.78 and results["precision"] == 0.75 and abs(results["recall"] - 0.783) < 0.001)
check("3. new_probs", lambda: abs(new_probs[0] - 0.466) < 0.005 and abs(new_probs[1] - 0.961) < 0.005)

### 🏋️ Exercise 7 · Practice — Several models side by side

In a notebook written as a series of cells, trying a different learning rate means changing `lr` and re-running the training cell, which replaces the old model. With a class, each model is a separate object, so you can keep several and compare them.

1. Complete `train(model, X, y, lr, n_epochs)`, which trains `model` with the loop from Exercise 5 and returns its loss history
2. `models`: a dict from each learning rate in `rates` to a new `Linear(2, 1)` model, trained with that learning rate
3. `histories`: a dict from each learning rate to the loss history that `train` returned

The cell then plots the three loss histories on the same axes. 🔮 **Predict** before you run it: what will each curve look like? (Think about what you saw when you changed the learning rate for linear regression in LELA60331 Week 4.)

In [ ]:
def train(model, X, y, lr, n_epochs):
    history = []
    for epoch in range(n_epochs):
        scores = ???                                   # 1.
        history.append(???)
        ???
        ???
    return history

rates = [0.01, 0.5, 2]
models = {}
histories = {}
for lr in rates:
    random.seed(10)
    models[lr] = ???                                         # 2.
    histories[lr] = ???                 # 3.

for lr in rates:
    plt.plot(histories[lr], label="lr = " + str(lr))
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()
for lr in rates:
    print(lr, round(histories[lr][-1], 3), evaluate(models[lr], X, y))

In [ ]:
# ✅ Check
check("1–2. three models", lambda: sorted(models) == rates and all(isinstance(m, Linear) for m in models.values()))
check("3. each trained for 1000 epochs", lambda: all(len(h) == 1000 for h in histories.values()))
check("lr = 0.5 gives the same as Exercise 5", lambda: abs(histories[0.5][-1] - loss_history[-1]) < 1e-9)
check("the first model is unchanged", lambda: abs(bce_loss(model.forward(X), y) - 0.656) < 0.001)

⭐ **Challenge (optional):** add a method `__str__` to `Linear` (Week 7, Part 3), so that `print(model)` shows its weights and biases, rounded to 2 decimal places.

---
## ✏️ Summary

You can now:

* explain why a program with many shared global names is fragile, and how a class helps
* write a `Linear` layer with a **forward** pass, a **backward** pass and an **update**
* write a loss as two functions: one for its value, and one for its gradient
* train a model with a loop of four lines: forward, loss, backward, update
* keep several models side by side, and evaluate them with functions that work for any model